
## <font color='blue'>Generative AI and LLMs for Natural Language Processing</font>
## <font color='blue'>Fine-Tuning an Open-Source LLM for a Legal Assistant App</font>

## Installing and Loading Packages

In [ ]:
# To update a package, run the command below in the terminal or command prompt:
# pip install -U package_name

# To install an exact version of a package, run the command below in the terminal or command prompt:
# !pip install package_name==desired_version

# After installing or updating a package, restart the Jupyter notebook.

# Install the watermark package.
# This package is used to record the versions of other packages used in this Jupyter notebook.
!pip install -q -U watermark

In [ ]:
# Installing packages
!pip install -q -r requirements.txt

In [ ]:
# Imports
import numpy as np
import nltk
import shutil
import evaluate
import transformers
import datasets
from datasets import load_dataset
from transformers import T5Tokenizer, DataCollatorForSeq2Seq
from transformers import T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Versions of packages used in this Jupyter notebook
%reload_ext watermark
%watermark -a "Lucas Gabriel"

In [ ]:
# Remove folders if they already exist. This avoids errors when running the notebook a second time.
try:
    shutil.rmtree('logs_treino')
    shutil.rmtree('resultados_treino')
    shutil.rmtree('modelo_salvo')
except:
    print('Folders do not exist or have already been removed!')

## Loading the Data

In [ ]:
# Set the file name
nome_arquivo = "dataset.csv"

In [ ]:
# Load the data
dsa_dataset = load_dataset('csv', data_files = nome_arquivo, delimiter = ',')

In [ ]:
# Train/test split with 80/20 ratio
dsa_dataset = dsa_dataset["train"].train_test_split(test_size = 0.2)

In [ ]:
# Inspect dataset format
dsa_dataset

## Tokenizer and Open-Source LLM

https://huggingface.co/google/flan-t5-base

In [ ]:
# Load tokenizer
tokenizador = T5Tokenizer.from_pretrained("google/flan-t5-base", legacy = False)

In [ ]:
# Inspect tokenizer
tokenizador

In [ ]:
# Load pre-trained LLM
modelo = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

In [ ]:
# Inspect LLM architecture
modelo

In [ ]:
# Data collator to combine tokenizer and model
data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizador, model = modelo)

In [ ]:
# Inspect data collator
data_collator

## Preprocessing

In [ ]:
# All inputs will receive the prefix: "answer the question:"
prefix = "answer the question: "

In [ ]:
# Preprocessing function
def dsa_fn_preprocess(data):

    # Concatenate the prefix to each question in the provided data["question"] list
    inputs = [prefix + doc for doc in data["question"]]

    # Use the tokenizer to convert processed questions into tokens with a maximum length of 128,
    # truncating those that are longer
    model_inputs = tokenizador(inputs, max_length = 128, truncation = True)

    # Tokenize the answers provided in data["answer"] with a maximum length of 512,
    # truncating those that are longer
    labels = tokenizador(text_target = data["answer"], max_length = 512, truncation = True)

    # Add the answer tokens as labels in the model input dictionary
    model_inputs["labels"] = labels["input_ids"]

    # Return the processed dictionary containing both tokenized inputs and labels
    return model_inputs

In [ ]:
# Apply preprocessing function to dataset, generating the tokenized dataset
dsa_dataset_tokenized = dsa_dataset.map(dsa_fn_preprocess, batched = True)

In [ ]:
# Inspect the tokenized dataset
dsa_dataset_tokenized

In [ ]:
dsa_dataset_tokenized['train']['question'][0]

In [ ]:
dsa_dataset_tokenized['train']['answer'][0]

In [ ]:
dsa_dataset_tokenized['train']['input_ids'][0]

In [ ]:
dsa_dataset_tokenized['train']['labels'][0]

## Defining the Model Evaluation Metric

In [ ]:
# The "punkt" package is specific to tokenization, which involves splitting text
# into a list of sentences
nltk.download("punkt", quiet = True)

In [ ]:
# Download supplemental punkt_tab module
nltk.download('punkt_tab')

In [ ]:
# Define metric
metric = evaluate.load("rouge")

In [ ]:
# Metric calculation function
def dsa_calcula_metricas(eval_preds):

    # Unpack predictions and labels from eval_preds
    previsoes, labels = eval_preds

    # Replace all -100 values in labels with the pad token ID
    labels = np.where(labels != -100,
                      labels,
                      tokenizador.pad_token_id)

    # Decode predictions to text, skipping special tokens
    previsoes_decodificadas = tokenizador.batch_decode(previsoes,
                                                       skip_special_tokens = True)

    # Decode labels to text, skipping special tokens
    labels_decodificados = tokenizador.batch_decode(labels,
                                                    skip_special_tokens = True)

    # Add a newline after each sentence in decoded predictions,
    # preparing them for ROUGE evaluation
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in previsoes_decodificadas]

    # Add a newline after each label in decoded labels,
    # preparing them for ROUGE evaluation
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in labels_decodificados]

    # Compute the ROUGE metric between decoded predictions and labels, using a stemmer
    resultado = metric.compute(predictions = previsoes_decodificadas,
                               references = labels_decodificados,
                               use_stemmer = True)

    # Return the ROUGE metric result
    return resultado

In [ ]:
# Define training arguments
dsa_training_args = Seq2SeqTrainingArguments(output_dir = "resultados_treino",
                                             eval_strategy = "epoch",
                                             learning_rate = 3e-4,
                                             logging_dir = "logs_treino",
                                             logging_steps = 1,
                                             per_device_train_batch_size = 4,
                                             per_device_eval_batch_size = 2,
                                             weight_decay = 0.01,
                                             save_total_limit = 3,
                                             num_train_epochs = 3,
                                             predict_with_generate = True,
                                             push_to_hub = False,
                                             report_to = "none")

In [ ]:
# Define the trainer
dsa_trainer = Seq2SeqTrainer(model = modelo,
                             args = dsa_training_args,
                             train_dataset = dsa_dataset_tokenized["train"],
                             eval_dataset = dsa_dataset_tokenized["test"],
                             tokenizer = tokenizador,
                             data_collator = data_collator,
                             compute_metrics = dsa_calcula_metricas)

## Model Training

In [ ]:
%%time
dsa_trainer.train()

In [ ]:
# Save the model
dsa_trainer.save_model('modelo_salvo')

**ROUGE-1** measures overlap of unigrams (individual words).

**ROUGE-2** measures overlap of bigrams (pairs of consecutive words).

**ROUGE-L** measures overlap of the longest common subsequence between the generated summary and the reference summary. This takes word order into account while allowing gaps. Higher values indicate better performance. ROUGE-L is computed based on sequence similarity, taking precision, recall, and their harmonic mean into account.

Higher ROUGE values indicate greater similarity between the generated summary and the reference summary, which is generally interpreted as higher summary quality. However, it is important to remember that no metric can fully capture the quality of a summary, and it is useful to complement evaluation with qualitative analysis or other metrics.

## Model Deployment and Usage

In [ ]:
# Load the final saved tokenizer from disk
tokenizador_dsa_final = AutoTokenizer.from_pretrained('modelo_salvo')

In [ ]:
# Load the final saved model from disk
modelo_dsa_final = AutoModelForSeq2SeqLM.from_pretrained('modelo_salvo')

In [ ]:
# Inspect the model architecture
modelo_dsa_final

In [ ]:
# Input text
texto_entrada = "Can I move out of state with my children if I have a custody agreement in that state?"

In [ ]:
# Tokenize the input
texto_entrada_tokenizado = tokenizador_dsa_final(texto_entrada, return_tensors = "pt").input_ids

In [ ]:
# Generate output (model prediction)
texto_saida_tokenizado = modelo_dsa_final.generate(texto_entrada_tokenizado,
                                                   max_length = 50,
                                                   temperature = 0.4,
                                                   do_sample = True)

In [ ]:
# Inspect output
texto_saida_tokenizado

In [ ]:
# Decode output
texto_saida = tokenizador_dsa_final.decode(texto_saida_tokenizado[0],
                                           skip_special_tokens = True)

In [ ]:
# Result
print("Question:", texto_entrada)
print("Answer:", texto_saida)

In [ ]:
%reload_ext watermark
%watermark -a "Lucas Gabriel"

In [ ]:
#%watermark -v -m

In [ ]:
#%watermark --iversions

# End